# File Manager Smoke Tests

Staged smoke tests for `FileManager`. These cells generate deterministic synthetic images, preview filenames, save TIFF data with `FrameMetaData`, reload and verify it, exercise custom filename patterns, and list saved files. Output is isolated under `/tmp` by default.

In [10]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "filemanager.py").is_file():
            return candidate
    raise RuntimeError("Could not find the EvoMachine repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np

from evomachine.coordinates import Coordinate
from evomachine.filemanager import FileManager, FileNameConfig
from evomachine.frame import FrameMetaData
from evomachine.types import FilterWheelType, LEDType


@dataclass(frozen=True, kw_only=True)
class FileManagerTestSettings:
    output_directory: Path = Path("/tmp/evomachine-file-manager-testing")
    image_width: int = 64
    image_height: int = 48
    random_seed: int = 1
    fov_id: int = 3


SETTINGS = FileManagerTestSettings()
SETTINGS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


FileManagerTestSettings(output_directory=PosixPath('/tmp/evomachine-file-manager-testing'), image_width=64, image_height=48, random_seed=1, fov_id=3)

## Create the manager and representative metadata

The output directory is created automatically. No camera or physical peripheral is required.

In [11]:
file_manager = FileManager(
    FileNameConfig(directory=SETTINGS.output_directory)
)
frame_metadata = FrameMetaData(
    frame_id=7,
    leds={LEDType.LED_450_NM: 25.0},
    filter_wheel=FilterWheelType.FILTER_465nm,
    exposure=20,
    fov_id=SETTINGS.fov_id,
    coordinate=Coordinate(10, 20, 30),
    callback_id=11,
    additional_metadata={"experiment": "file-manager-smoke-test"},
)
rng = np.random.default_rng(SETTINGS.random_seed)
test_image = rng.integers(
    low=0,
    high=np.iinfo(np.uint16).max + 1,
    size=(SETTINGS.image_height, SETTINGS.image_width),
    dtype=np.uint16,
)

{
    "output_directory": file_manager.config.directory,
    "directory_exists": file_manager.config.directory.is_dir(),
    "image_shape": test_image.shape,
    "image_dtype": test_image.dtype,
}

{'output_directory': PosixPath('/tmp/evomachine-file-manager-testing'),
 'directory_exists': True,
 'image_shape': (48, 64),
 'image_dtype': dtype('uint16')}

## Preview the generated filename

This renders channel, field of view, coordinate, filter, and timestamp fields without writing an image.

In [12]:
preview_path = file_manager.get_filename(frame_metadata)
assert preview_path.parent == SETTINGS.output_directory
assert preview_path.suffix == ".tiff"
preview_path

PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_16-20-08-545574+0100.tiff')

## Save and verify one TIFF

`save_frame` writes the NumPy image and serialised `FrameMetaData`. The reload assertions verify both pixel fidelity and important metadata fields.

In [13]:
saved_path = file_manager.save_frame(
    frame=test_image,
    frame_metadata=frame_metadata,
)
image_only = FileManager.load_image(saved_path)
metadata_only = FileManager.load_tiff_metadata(saved_path)
loaded_image, loaded_metadata = FileManager.load_frame(saved_path)
saved_metadata = loaded_metadata["FrameMetaData"]

assert np.array_equal(image_only, test_image)
assert metadata_only == loaded_metadata
assert np.array_equal(loaded_image, test_image)
assert saved_metadata["frame_id"] == frame_metadata.frame_id
assert saved_metadata["fov_id"] == frame_metadata.fov_id
assert saved_metadata["coordinate"] == {"X": 10, "Y": 20, "Z": 30}
assert saved_metadata["leds"]["LED_450_NM"]["brightness"] == 25.0
assert saved_metadata["additional_metadata"]["experiment"] == "file-manager-smoke-test"

{
    "saved_path": saved_path,
    "saved_bytes": saved_path.stat().st_size,
    "pixels_match": np.array_equal(loaded_image, test_image),
    "execution_time": saved_metadata["execution_time"],
}

{'saved_path': PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_16-20-16-616397+0100.tiff'),
 'saved_bytes': 6912,
 'pixels_match': True,
 'execution_time': '2026-07-02T16:20:16.616397+01:00'}

## Exercise configuration and output-directory methods

This checks `copy`, `updated`, `update_from_mapping`, complete config replacement, incremental manager updates, automatic directory creation, and use of an existing directory with creation disabled.

In [14]:
base_config = FileNameConfig(directory=SETTINGS.output_directory)
copied_config = base_config.copy()
updated_config = base_config.updated(filename_pattern="{channel}_F{frame_id}")
mapped_config = base_config.update_from_mapping(
    {"directory": SETTINGS.output_directory / "mapped-output"}
)

assert copied_config == base_config and copied_config is not base_config
assert updated_config.filename_pattern == "{channel}_F{frame_id}"
assert base_config.filename_pattern != updated_config.filename_pattern

file_manager.update_config(config=mapped_config)
assert mapped_config.directory.is_dir()
file_manager.update_config(
    directory=SETTINGS.output_directory,
    filename_pattern=base_config.filename_pattern,
)
existing_directory_config = FileNameConfig(
    directory=SETTINGS.output_directory,
    create_directory=False,
)
existing_directory_manager = FileManager(existing_directory_config)

{
    "copy_is_independent": copied_config is not base_config,
    "mapped_directory_created": mapped_config.directory.is_dir(),
    "active_directory": file_manager.config.directory,
    "existing_directory_accepted": existing_directory_manager.config.directory.is_dir(),
}

{'copy_is_independent': True,
 'mapped_directory_created': True,
 'active_directory': PosixPath('/tmp/evomachine-file-manager-testing'),
 'existing_directory_accepted': True}

## Check filename edge cases

Missing optional values receive stable placeholders, and multiple LED channels use a filename-safe period separator.

In [15]:
minimal_metadata = FrameMetaData(
    frame_id=8,
    leds=None,
    filter_wheel=None,
    exposure=None,
)
multi_led_metadata = FrameMetaData(
    frame_id=9,
    leds={LEDType.LED_450_NM: 25.0, LEDType.LED_565_NM: 30.0},
    filter_wheel=None,
    exposure=20,
)
minimal_path = file_manager.get_filename(minimal_metadata)
multi_led_path = file_manager.get_filename(multi_led_metadata)
assert minimal_path.name.startswith("NO_LED_FOV-1_X_Y_Zauto_FNone_")
assert multi_led_path.name.startswith("LED450NM.LED565NM_")

{"minimal": minimal_path, "multiple_leds": multi_led_path}

{'minimal': PosixPath('/tmp/evomachine-file-manager-testing/NO_LED_FOV-1_X_Y_Zauto_FNone_2026-07-02_16-20-36-706401+0100.tiff'),
 'multiple_leds': PosixPath('/tmp/evomachine-file-manager-testing/LED450NM.LED565NM_FOV-1_X_Y_Zauto_FNone_2026-07-02_16-20-36-706443+0100.tiff')}

## Exercise a custom filename pattern

Additional metadata may be referenced directly by the filename pattern. Image-like suffixes are normalised to `.tiff`.

In [16]:
file_manager.update_config(
    filename_pattern="{experiment}/{channel}_FOV{fov_id}_frame{frame_id}.png",
)
custom_path = file_manager.save_frame(test_image, frame_metadata)
assert custom_path.suffix == ".tiff"
assert custom_path.parent.name == "file-manager-smoke-test"
custom_path

PosixPath('/tmp/evomachine-file-manager-testing/file-manager-smoke-test/LED450NM_FOV3_frame7.tiff')

## List saved TIFF files

The listing helper accepts a relative glob and returns sorted file paths.

In [17]:
top_level_files = FileManager.list_filenames(
    directory=SETTINGS.output_directory,
    filename_pattern="*.tiff",
)
nested_files = FileManager.list_filenames(
    directory=SETTINGS.output_directory,
    filename_pattern="**/*.tiff",
)
{
    "top_level_files": top_level_files,
    "all_tiff_files": nested_files,
    "count": len(nested_files),
}

{'top_level_files': [PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-37-17-496906+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-40-35-018646+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-45-47-481368+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-46-48-985960+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_16-20-16-616397+0100.tiff')],
 'all_tiff_files': [PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-37-17-496906+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-40-35-018646+0100.tiff'),
  PosixPath('/tmp/evomachine-file-manager-testing/LED450NM_FOV3_X10_Y20_Z30_F1_2026-07-02_12-45-47-481368+0100.tiff'),
  PosixPat

## Exercise guarded validation failures

These checks deliberately pass invalid values and confirm that the public API raises the documented exception instead of silently accepting them.

In [18]:
from tempfile import TemporaryDirectory


def captured_exception(expected_type: type[Exception], action) -> str:
    try:
        action()
    except expected_type as error:
        return f"{type(error).__name__}: {error}"
    raise AssertionError(f"Expected {expected_type.__name__}.")


validation_results = {}
validation_results["empty_pattern"] = captured_exception(
    ValueError,
    lambda: FileNameConfig(directory=SETTINGS.output_directory, filename_pattern=""),
)
validation_results["unknown_config_field"] = captured_exception(
    ValueError,
    lambda: file_manager.config.updated(not_a_field=True),
)
validation_results["wrong_metadata_type"] = captured_exception(
    TypeError,
    lambda: file_manager.get_filename("not metadata"),
)
validation_results["wrong_frame_type"] = captured_exception(
    TypeError,
    lambda: file_manager.save_frame([[1, 2]], frame_metadata),
)
validation_results["absolute_glob"] = captured_exception(
    ValueError,
    lambda: FileManager.list_filenames(SETTINGS.output_directory, "/absolute/*.tiff"),
)
validation_results["missing_image"] = captured_exception(
    FileNotFoundError,
    lambda: FileManager.load_image(SETTINGS.output_directory / "missing.tiff"),
)
with TemporaryDirectory() as temporary_directory:
    missing_directory = Path(temporary_directory) / "missing"
    validation_results["creation_disabled"] = captured_exception(
        FileNotFoundError,
        lambda: FileManager(
            FileNameConfig(directory=missing_directory, create_directory=False)
        ),
    )

validation_results

{'empty_pattern': 'ValueError: FileNameConfig: filename_pattern must not be empty.',
 'unknown_config_field': "ValueError: FileNameConfig.updated: unknown fields ['not_a_field'].",
 'wrong_metadata_type': "TypeError: FileManager.get_filename: frame_metadata must be FrameMetaData, received <class 'str'>.",
 'wrong_frame_type': "TypeError: FileManager.save_frame: frame must be np.ndarray, received <class 'list'>.",
 'absolute_glob': 'ValueError: FileManager.list_filenames: filename_pattern must be relative.',
 'missing_image': 'FileNotFoundError: FileManager.load_image: path does not exist: /tmp/evomachine-file-manager-testing/missing.tiff.',
 'creation_disabled': 'FileNotFoundError: FileManager: directory does not exist: /tmp/tmpl_68z5zb/missing.'}

## Cleanup guidance

Files are deliberately retained for inspection. Remove `/tmp/evomachine-file-manager-testing` manually when they are no longer needed.